# Image Evals Are Simpler Than You Think

In this video, we grade images that models made, and each grade is one more column next to its image, in the tables that already hold the images.

The method comes from OpenAI's [image evals cookbook](https://developers.openai.com/cookbook/examples/multimodal/image_evals), which grades an image in three steps:

1. Hard gates
2. Scores from 0 to 5
3. Tags for how an image failed

To run it, OpenAI built what they call a vision eval harness.

Image models fail quietly. A bad image looks finished until someone looks at it closely.

The usual eval is a script next to the app. Someone has to keep it running, and new images go ungraded until it runs again.

We'll use Pixeltable, a backend for AI apps on images and video, where a column can call a model. There are four recipes, and the code is in the repo linked below.

## Setup

Set your OpenAI and Gemini keys in your shell, and use this folder's `.venv` as the notebook's kernel. `uv sync` installs the versions this repo pins.

In [ ]:
!uv sync

## The judge is one more column next to the image

The tables are declared in `schema.py`. In `generations`, you fill in every column above `image`. `gen_prompt` goes to the image model, and `prompt` goes to the judge. Pixeltable computes `image` by calling OpenAI's image model.

In [ ]:
!sed -n '35,51p' schema.py

The grade lives in `gen_evals`, which holds only the `generations` rows where `eval_ready` is true. Its `judge_raw` column sends each image to GPT-5.2, and `scores` reads that one answer. For the flyer, `extracted_text` is a second model call.

In [ ]:
!sed -n '77,103p' schema.py

`pxt schema diff` shows what will change, and `pxt schema update` creates the tables.

In [ ]:
!pxt schema diff schema.py img_eval

In [ ]:
!pxt schema update schema.py img_eval -f

## Recipe 1: Mark images ready, and the judge grades the ones already in the table

The seed script inserts four rows, with prompts and criteria from each job folder's `examples.json`, so the image models make two images and two edits. That's four paid image calls, so it takes a little while. It leaves `eval_ready` false, so nothing gets graded yet.

In [ ]:
!python scripts/seed_demo.py

In [ ]:
import udfs  # register the judge functions before opening the tables
import pixeltable as pxt

generations = pxt.get_table('img_eval/generations')
edits = pxt.get_table('img_eval/edits')
gen_evals = pxt.get_table('img_eval/gen_evals')
edit_evals = pxt.get_table('img_eval/edit_evals')

generations.select(generations.image, generations.case_id, generations.workflow).collect()

Here are the checkout screen and the flyer, with no grades yet.

### The judge's prompt is built from the row

Before we pay for a grade, let's look at what the judge gets. `judge_user_text` fills in the row's `prompt` and `criteria`. The row's `workflow` picks the rubric, which gives the instructions and the JSON schema.

In [ ]:
!sed -n '92,112p' udfs.py

Here's that prompt filled in for the checkout screen. We run the same functions as a query, so no model gets called.

In [ ]:
row = generations.where(generations.case_id == 'ui_checkout_mockup').select(
    instructions=udfs.judge_instructions(generations.workflow),
    user_text=udfs.judge_user_text(generations.prompt, generations.criteria),
    text_format=udfs.judge_text_format(generations.workflow),
).collect()[0]

print(row['instructions'][:330], '...\n')
print(row['user_text'], '\n')
row['text_format']

The instructions come from the checkout rubric, and the user text is this row's prompt and criteria. The schema sets the fields the judge returns, and the image goes in next to the text.

Marking the images ready is one `update`. The judge grades both images already in the table, and no image gets made again.

In [ ]:
generations.update({'eval_ready': True}, where=generations.batch_id == 'demo_v1')

gen_evals.select(
    gen_evals.image, gen_evals.case_id, gen_evals.verdict,
    gen_evals.scores, gen_evals.tags,
).collect()

Each image gets a verdict, and `scores` holds the judge's numbers and its reason. The tags name how an image failed, so you can count failures by kind.

## Recipe 2: The flyer's exact words are a hard gate

A judge can miss a word, so the flyer's exact words are a hard gate. A model reads each line of text off the flyer, and plain Python compares those lines with `required_text`.

In [ ]:
gen_evals.where(gen_evals.workflow == 'marketing_flyer').select(
    gen_evals.extracted_text, gen_evals.text_check
).collect()

`text_check` lists any required lines the flyer is missing, and any extra text. A missing line fails the flyer, whatever the judge scored.

## Recipe 3: An edit is graded against the images it started from

Edits work the same way. In `edits`, you fill in the reference images and the instruction, and Pixeltable computes `image` by calling Gemini.

In [ ]:
!sed -n '54,74p' schema.py

The edit judge also needs to see what the edit started from. `edit_judge_content` puts the reference images in the judge's message, ahead of the edited result.

In [ ]:
!sed -n '106,118p' schema.py

In [ ]:
!sed -n '67,89p' udfs.py

Gemini made two edits: the logo, from one reference image, and the try-on, from two. Marking them ready grades both.

In [ ]:
edits.update({'eval_ready': True}, where=edits.batch_id == 'demo_v1')

edit_evals.select(
    edit_evals.image, edit_evals.case_id, edit_evals.verdict,
    edit_evals.scores, edit_evals.tags,
).collect()

Each edit gets a verdict and its scores, and the tags name what went wrong, if anything.

A new edit gets made and graded in one insert. This instruction changes the line under the name, which the seed edit left alone.

In [ ]:
logo_prompt = (
    'Edit the logo by changing the text LOCAL BAKERY to FRESH DAILY.\n'
    'Do not change any other text, colors, shapes, or layout.'
)

edits.insert(
    case_id='logo_tagline_edit',
    batch_id='run_2',
    workflow='logo_edit',
    gen_prompt=logo_prompt,
    prompt=logo_prompt,
    criteria=(
        'The requested edit is applied exactly.\n'
        'All non-target content remains unchanged.\n'
        'Character style, color, and geometry remain consistent with the original.'
    ),
    ref_image='use_cases/logo_edit/assets/logo_input.png',
    eval_ready=True,
)

edit_evals.where(edit_evals.case_id == 'logo_tagline_edit').select(
    edit_evals.ref_image, edit_evals.image, edit_evals.verdict, edit_evals.scores, edit_evals.tags
).collect()

The insert made the edit and graded it. The reference image sits next to the result, so you can check the judge yourself.

## Recipe 4: Tighten a rubric, and only the verdicts rerun

The judge gives the scores, and the rubric's Python turns them into a verdict. Right now the logo rubric passes an edit when every score is 4 or higher.

In [ ]:
!sed -n '140,155p' use_cases/logo_edit/rubric.py

We'll tighten it, so every score has to be a 5. An edit that passed with a 4 should fail now. I'm editing the file from Python so it's on screen, but you'd just open `rubric.py`.

In [ ]:
from pathlib import Path

rubric = Path('use_cases/logo_edit/rubric.py')
rubric.write_text(rubric.read_text().replace('or 0) < 4', 'or 0) < 5'))
!git diff use_cases/logo_edit/rubric.py

Python loaded the rubric when the kernel started, so restart the kernel to pick up the edit.

### Restart the kernel, and every image and grade is still there

The notebook has no variables now. Open the tables again, and every image and grade is still there, because they're stored in the tables, not in the notebook.

In [ ]:
import udfs  # register the judge functions before opening the tables
import pixeltable as pxt

edit_evals = pxt.get_table('img_eval/edit_evals')

edit_evals.select(edit_evals.image, edit_evals.case_id, edit_evals.verdict, edit_evals.tags).collect()

The threshold is plain Python, so only `verdict` and `tags` need to rerun. We note the table's version first, so we can read the old grades afterward.

In [ ]:
before = edit_evals.get_metadata()['version']

status = edit_evals.recompute_columns('verdict', 'tags')
status.updated_cols

Only `verdict` and `tags` were updated. Every update makes a new version of the table, and `pxt history` lists them.

In [ ]:
!pxt history img_eval/edit_evals -n 3

An older version opens by its number. Here are the old verdicts next to the new ones, and the judge's response IDs show it's the same answer.

In [ ]:
old = pxt.get_table(f'img_eval/edit_evals:{before}')
old_rows = old.select(
    old.case_id, old_verdict=old.verdict, old_response=old.judge_raw['id']
).collect().to_pandas()
new_rows = edit_evals.select(
    edit_evals.case_id, edit_evals.verdict, edit_evals.tags, response=edit_evals.judge_raw['id']
).collect().to_pandas()

both = old_rows.merge(new_rows, on='case_id')
both['same_judge_answer'] = both.old_response == both.response
both[['case_id', 'old_verdict', 'verdict', 'tags', 'same_judge_answer']]

Any edit that passed with a 4 fails now. `same_judge_answer` is true on every row, so the judge didn't run again and no image got made.

A change to the judge's prompt or JSON schema is different. That reruns `judge_raw`, a judge call on every row, and everything under it.

In [ ]:
!pxt recompute img_eval/edit_evals judge_raw -n

With `-n`, it only reports what would rerun. Three rows means three judge calls, and the images still don't get made again.

## Put the rubric back and delete the sample rows

To finish, put the rubric back the way it was and delete the rows we added.

In [ ]:
!git checkout -- use_cases/logo_edit/rubric.py

generations = pxt.get_table('img_eval/generations')
edits = pxt.get_table('img_eval/edits')
generations.delete(where=generations.batch_id.isin(['demo_v1', 'run_2']))
edits.delete(where=edits.batch_id.isin(['demo_v1', 'run_2']))

## Every grade sits next to its image

We graded two images and three edits, and every grade sits in a column next to its image, in the tables that already hold the images.

| Recipe | What gets graded | What runs |
|---|---|---|
| 1. Mark images ready | the checkout screen and the flyer, already in the table | the judge, with no new images |
| 2. A hard gate on exact words | the flyer's text, line by line | a model reads the text, and plain Python compares it |
| 3. Edits | the logo and the try-on, then a new logo edit on insert | Gemini makes the edit, and the judge sees it next to its references |
| 4. A tighter rubric | the same edits, verdicts only | only `verdict` and `tags`, with no judge calls |

Each job folder in the repo has its own `rubric.py`, so start from the one closest to your images. A star on the Pixeltable repo helps other developers find the project.